In [0]:
import pyspark.sql.functions as F
SCHEMA = "workspace.erp_lakehouse"

In [0]:
def clean(col):
    # lowercase, drop punctuation, collapse spaces
    c = F.lower(F.col(col))
    c = F.regexp_replace(c, r"[^a-z0-9 ]", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

b_items = (spark.table(f"{SCHEMA}.silver_b_items")
    .withColumn("b_key", clean("part_desc"))
    .select("part_no", "part_desc", "b_key"))

bc_items = (spark.table(f"{SCHEMA}.bronze_bc_items")
    .withColumn("bc_key", clean("displayName"))
    .select(F.col("number").alias("bc_item_no"), "displayName", "bc_key"))

# 1. Exact match after cleaning
exact = (b_items.join(bc_items, b_items.b_key == bc_items.bc_key, "inner")
    .withColumn("method", F.lit("exact after cleaning"))
    .withColumn("score", F.lit(1.0)))

# 2. Fuzzy match on whatever is left
left_b = b_items.join(exact.select("part_no"), "part_no", "left_anti")
left_bc = bc_items.join(exact.select("bc_item_no"), "bc_item_no", "left_anti")

fuzzy = (left_b.crossJoin(left_bc)
    .withColumn("dist", F.levenshtein("b_key", "bc_key"))
    .withColumn("score", 1 - F.col("dist") / F.greatest(F.length("b_key"), F.length("bc_key")))
    .filter("score >= 0.8")
    .withColumn("method", F.lit("fuzzy")))

# 3. Combine and keep the columns the crosswalk needs
cols = ["bc_item_no", "part_no", "displayName", "part_desc", "method", "score"]
xref_item = exact.select(*cols).unionByName(fuzzy.select(*cols))

xref_item.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.xref_item")
xref_item.display()
print(xref_item.count())